In [7]:
import pandas as pd
import json
import yaml
import seaborn as sns
from pathlib import Path

In [8]:
folder = Path('..', 'outputs', '004.experiment')

experiments = [ p for p in folder.iterdir()
                if p.is_dir() and Path(p, 'eval_results.json').exists() ]

for p in experiments: print(p.name)

49c4aca27fc210193fed546f2da4d95e
243fedb3306e6c38d547ce33ee5b920a
e42451505c6d52cf2dc20028c0d77ffa
bdc05806f1727bd97c26f84bda53ad23
e13bf9a149e1cdc5e78a39d5db17b031


In [9]:
def get_params(dir):
    params_file = Path(dir, 'parameters.yaml')
    parameters = yaml.safe_load(params_file.read_text())
    # parameters['folder'] = dir.name
    return parameters

def read_json_results(folder, base_file):
    file = Path(folder, base_file)
    results = json.loads(file.read_text())
    results['folder'] = folder.name
    return results

df_params = pd.DataFrame([ get_params(p) for p in experiments ])
df_eval = pd.DataFrame([read_json_results(e, 'eval_results.json') for e in experiments])
df_valids = pd.DataFrame([read_json_results(e, 'valid_results.json') for e in experiments])

df = pd.merge(df_params, df_eval, left_on='_hash_id', right_on='folder')

teacher_keys = df['teachers_keys'].loc[0] # ['t5', 'llama']

df = pd.concat([
    df,
    df['teachers_weights'].apply(pd.Series, index=teacher_keys)
], axis=1)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 36 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   _hash_id                 5 non-null      object 
 1   _timestamp               5 non-null      object 
 2   batch_size               5 non-null      int64  
 3   bf16                     5 non-null      bool   
 4   dataset                  5 non-null      object 
 5   eval_steps               5 non-null      int64  
 6   experiment               5 non-null      object 
 7   from_pretrained          5 non-null      object 
 8   generation_max_length    5 non-null      int64  
 9   grad_steps               5 non-null      int64  
 10  local_rank               5 non-null      int64  
 11  logging_strategy         5 non-null      object 
 12  lora_rank                5 non-null      int64  
 13  lr                       5 non-null      float64
 14  max_input_length         5 non

In [10]:
df_valids

,epoch,eval_accuracy,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,eval_token_accuracy,folder
0,30.042918,0.742,0.021144,46.1844,10.826,0.173,0.83454,49c4aca27fc210193fed546f2da4d95e
1,7.526882,0.748,0.012867,55.9277,8.940,0.572,0.81802,243fedb3306e6c38d547ce33ee5b920a
2,59.829060,0.728,0.023825,21.9353,22.794,0.182,0.84898,e42451505c6d52cf2dc20028c0d77ffa
3,3.765465,0.756,0.009986,79.3824,6.299,0.794,0.81440,bdc05806f1727bd97c26f84bda53ad23
4,15.053763,0.748,0.017203,38.7363,12.908,0.413,0.82338,e13bf9a149e1cdc5e78a39d5db17b031


In [11]:
df[
    ['dataset', '_timestamp', 'batch_size', 'lr', 'lora_rank', 'eval_accuracy', 'eval_token_accuracy', 'eval_loss']
].sort_values(by='eval_accuracy', ascending=False)

,dataset,_timestamp,batch_size,lr,lora_rank,eval_accuracy,eval_token_accuracy,eval_loss
1,obqa,2025-11-17T02:41:54.157507,16,0.0005,16,0.706,0.91650,0.016763
4,obqa,2025-11-17T05:34:32.048467,32,0.0005,16,0.704,0.91740,0.019305
0,obqa,2025-11-17T10:19:41.044635,64,0.0005,16,0.692,0.92022,0.027306
3,obqa,2025-11-17T00:54:07.652086,8,0.0005,16,0.686,0.91586,0.013615
2,obqa,2025-11-17T19:21:07.919193,128,0.0005,16,0.682,0.92594,0.027009
